In [260]:
import random
import math
import re

In [282]:
class Lab01:

    def __init__(self):
        self.next_word_counts = {}

    # Принимает список слов обучающего набора
    def fit(self, words):

        for i in range(len(words)-1):
            current_word = words[i]
            next_word = words[i+1]

            if not self.next_word_counts.get(current_word):
                self.next_word_counts[current_word] = {}


            self.next_word_counts[current_word][next_word] = \
                self.next_word_counts[current_word].get(next_word, 0) + 1    

    # Вычисляет вероятность встретить слово w после слова v
    def prob(self, v, w, alpha=0):
        if v not in self.next_word_counts:
            vw_count = 0
            v_count = 0
        else:
            vw_count = self.next_word_counts[v].get(w, 0)
            v_count = sum(self.next_word_counts[v].values())

        if v_count == 0 and alpha == 0:
            return 0

        v_max = len(self.next_word_counts) # Количество уникальных слов, размер словаря

        # В формуле мы используем сглаживание Лапласа, т.к. слов, которых не было в обучающей выборке
        # мы будем получать 0
        return (
            (vw_count + alpha) / (v_count + alpha * v_max)
        )

    # Вычисляет сумму вероятностей, пропущенных через логарифм для заданного текста
    def logprob(self, text: list, alpha=1):

        sum_log_p = 0
        for i in range(len(text)-1):
            
            # Здесь мы передаем параметр сглаживания alpha - чтобы не получать 0 в логарифме
            sum_log_p += math.log(self.prob(text[i], text[i+1], alpha))
        return sum_log_p

    # Создает текст с заданной длиной
    def generate(self, word, count_words, T):

        text = word + ' '
        word = word.lower()

        for i in range(count_words):

            next_words_probability = []

            if not self.next_word_counts.get(word):
                print('Нет такого слова')
                break

            next_words = list(self.next_word_counts[word].keys())
            next_probabilities = []

            for next_word in next_words:
                pr = self.prob(v=word, w=next_word)
                next_probabilities.append(pr**(1/T))

            # Нормализуем вероятности, чтобы их сумма была = 1
            sum_before_normalization = sum(next_probabilities)
            next_probabilities = [x / sum_before_normalization for x in next_probabilities]

            next_word_final = random.choices(next_words, weights=next_probabilities, k=1)[0]
            text += next_word_final +  ' '
            word = next_word_final

        text = re.sub(r'\s+([.,!?;:])', r'\1', text)
        return text


## Корпус и токенизация

Обрабатываем исходный текст

In [265]:
with open('C:\Projects\\tgo_course\Oblomov.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# Переводим в нижний регистр
text = text.lower()

# Ищем слова и выбранные знаки препинания
words = re.findall(r'\w+(?:-\w+)*|[.,!?]', text)

# Делим текст на выборку: 90% train и 10% test
words_test_split = words[int(len(words)*0.9):]
words_train_split = words[:int(len(words)*0.9)]

print(len(words_test_split))
print(len(words_train_split))

18227
164043


Обучаем модель

In [267]:
lab01 = Lab01()

print(words[10000:10020])
lab01.fit(words_train_split)

[',', 'илья', 'ильич', '.', 'куда', 'это', '?', 'да', 'к', 'овчинину-то', ',', 'поедемте', '.', 'там', 'матвей', 'андреич', 'альянов', ',', 'казимир', 'альбертыч']


## Счетчики и вероятности

Смотрим вероятность пары слов на нескольких примерах

In [273]:
print(lab01.prob('может', 'быть'))
print(lab01.prob('завтра', 'будет'))
print(lab01.prob('пойдем', 'к'))
print(lab01.prob('тебе', '?'))

0.6948051948051948
0.010752688172043012
0.037037037037037035
0.052941176470588235


## Генерация

Генерируем 5 предложений по 30 слов с разной температурой T. Можно заметить, что с увеличением T повышается и разнообразие текста - появляются более редки комбинации слов. С уменьшением же T модель генерирует более вероятные слова, текст становится более "сухим"

In [274]:
print('T=1', lab01.generate('кровать', 30, 1))
print('T=2', lab01.generate('кровать', 30, 2))
print('T=4', lab01.generate('кровать', 30, 4))
print('T=0.5', lab01.generate('кровать', 30, 0.5))
print('T=0.25', lab01.generate('кровать', 30, 0.25))

T=1 кровать тюфяки, а я и штольц и отпустил на свете одинаково согласно и никаким уважением, что в ее голову фуражку и как вы не мог уже никакая пушка не 
T=2 кровать тюфяки, любить обломова! закричал он учился любви превращался в особом ящике комода, чищенье сапог обломова, об имении ольги открывалась ему явлений окружающей. они вора, 
T=4 кровать тюфяки, ростом и какие сильные пружины, косо, дам завтра пошлю к устройству собственного промаха. какой ты ограблен кругом и селянку? сейчас докладную записку к лошадям 
T=0.5 кровать тюфяки, и не знаю, не может быть, что ли, что у него, что он, как будто он, что у вас не может быть 
T=0.25 кровать тюфяки, что я не было, что если б не было, что она, что он, что он, что ж ты, что ж ты не 


In [275]:
print('T=1', lab01.generate('Дорога', 30, 1))
print('T=2', lab01.generate('Дорога', 30, 2))
print('T=4', lab01.generate('Дорога', 30, 4))
print('T=0.5', lab01.generate('Дорога', 30, 0.5))
print('T=0.25', lab01.generate('Дорога', 30, 0.25))

T=1 Дорога! опять одна монотонно произнес обломов, мирно-хлопотливой жизни больного! о чем? как встрепанный. дай вспомнить casta diva. это все едут по дороге. там жили 
T=2 Дорога ну? перебила она понюхала пятно на дороге, работает, направляясь в ход. ей глядеть на выборы, установить, измято и там жители ходили с ее рояля 
T=4 Дорога ну что без борьбы смущения и разбойников, открывая круглую табакерку. ой ли привыкнешь? повторил сын. остановившись, работает творческий ум ольги она мало этого безмолвного, 
T=0.5 Дорога, в глазах а если б я не без всяких хлопот много, а не было, без этого, что ж вы, и не было неловко, что 
T=0.25 Дорога, что я не было, что это, что это?. он, что я не было, что это? спросил обломов. он, не стану 


In [276]:
print('T=1', lab01.generate('Может', 30, 1))
print('T=2', lab01.generate('Может', 30, 2))
print('T=4', lab01.generate('Может', 30, 4))
print('T=0.5', lab01.generate('Может', 30, 0.5))
print('T=0.25', lab01.generate('Может', 30, 0.25))

T=1 Может быть, боже мой! она нагибалась и быстро начертил по деревне, куда же легко ли? даром что никакой удар не то сесть! разве кто-нибудь в детстве 
T=2 Может быть дома покойника головой. ковры да, шутка не спится, расцветание начали пропадать свой мимо барина за пятки, постой дай забыть не иметь конец торчит. так 
T=4 Может подумать вслух. путь! приставал в деда, маешься до испуга, исходить швейцарию, добрый бурш будет сиять ольга невольно сменился взглядом, представив себе новую квартиру! 
T=0.5 Может быть, так и не знаю, а где же, и не могу, не будет, и не на него, и что это не было, как 
T=0.25 Может быть, что ж ты, что я не было, что он. что он, что ж это? да, что ж ты, что ж ты 


In [277]:
word = 'Есть'
print('T=1', lab01.generate(word, 30, 1))
print('T=2', lab01.generate(word, 30, 2))
print('T=4', lab01.generate(word, 30, 4))
print('T=0.5', lab01.generate(word, 30, 0.5))
print('T=0.25', lab01.generate(word, 30, 0.25))

T=1 Есть вещи ах! я пойду к принятию каких-нибудь мер. что я не решался. да, говорил илья ильич. он и преподана им бежало облако непроницаемости слетело с 
T=2 Есть обязательная привычка? своя, из посторонних, отняв руку от лица только похоронная процессия обратит на свежем лице прибавилось что-то такое случилось давно предвидела это несправедливо. летом отдохну 
T=4 Есть что-нибудь необыкновенное, поблекли, увидя покойника, догадался артемий. дай-ка, удивляетесь сердитесь ну их они первые заметили у министра, прежде троекратного потчеванья и легли вповалку, 
T=0.5 Есть счастливые люди, как будто ушла. я не выйти из дома, да, как хорошо, а он в нем. ну, что за лес и теперь 
T=0.25 Есть, что ж ты? спросил обломов. он, что ж ты, и не было, что я не знаю, что я не было, и не 


In [279]:
word = 'будешь'
print('T=1', lab01.generate(word, 30, 1))
print('T=2', lab01.generate(word, 30, 2))
print('T=4', lab01.generate(word, 30, 4))
print('T=0.5', lab01.generate(word, 30, 0.5))
print('T=0.25', lab01.generate(word, 30, 0.25))

T=1 будешь через год секретарем. обломов, глаза так рано, другой день кофе к обеду, а магарыч-то какой еще и сами можете, сказал вам утром в девичью, 
T=2 будешь кататься в грозной, ах вы оставили даже комплимента для кого-нибудь третьего, андрей. все они странствовали по голосу своего рода неудобства, ждут меня нет огня, на 
T=4 будешь вперед не забыл. освободясь от такого холодного воззрения на трое, тебя, пустой, разбил? родственник. заселилось воображение и разразились хохотом. другой лежал илья иванович 
T=0.5 будешь делать, и не было! с полей и в последний раз, сказав, как будто не в деревню, что я не было, что, а вы 
T=0.25 будешь считать ненужным женщине, что ж ты не было, что он. а я не могу, что я не было, а там, и не было, 


## Правдоподобие текста

Взяли 50 слов из отложенной части - это осмысленный кусок текста. Затем тот же текст подали в модель в перемешку. Как результат - второй тест показал меньшее правдоподобие

In [256]:
words_for_test = words_test_split[:50]
print(words_for_test)
print(lab01.logprob(words_for_test))

random.shuffle(words_for_test)
print(words_for_test)
print(lab01.logprob(words_for_test))

['половину', 'состояния', 'давай', 'вексель', 'на', 'его', 'имя', 'ты', 'теперь', 'не', 'пьян', ',', 'в', 'своем', 'уме', ',', 'давай', ',', 'говорю', 'тебе', ',', 'я', 'без', 'того', 'не', 'выйду', 'что', 'вы', ',', 'михей', 'андреич', ',', 'кричите', 'так', '?', 'сказали', 'хозяйка', 'и', 'анисья', ',', 'выглянув', 'из-за', 'дверей', '.', 'двое', 'прохожих', 'остановились', ',', 'слушают', ',']
-439.653921600745
['анисья', 'теперь', 'михей', 'не', '?', ',', ',', ',', 'состояния', 'так', 'пьян', 'вы', 'прохожих', ',', 'его', 'двое', 'в', 'давай', 'и', 'без', 'из-за', 'слушают', 'выглянув', ',', ',', 'не', 'андреич', 'половину', 'ты', 'имя', '.', ',', ',', ',', 'того', 'остановились', 'давай', 'кричите', 'я', 'дверей', 'говорю', 'своем', 'на', 'тебе', 'сказали', 'вексель', 'что', 'выйду', 'хозяйка', 'уме']
-482.68974932524463


Тест работает, т.к. в модели учтено сглаживание Лапласа (см. метод logprob() и prob())

## Оценка

Посчитаем перплексию на отложенном тестовом куске текста

In [257]:
words_for_test = words_test_split
print(words_for_test[:20])
sm = lab01.logprob(words_for_test)
N = len(words_for_test) - 1

perplexity = math.exp(-sm/N)
print(perplexity)

['половину', 'состояния', 'давай', 'вексель', 'на', 'его', 'имя', 'ты', 'теперь', 'не', 'пьян', ',', 'в', 'своем', 'уме', ',', 'давай', ',', 'говорю', 'тебе']
7027.200590121818


Посмотри перплексию с разным параметром сглаживания

In [258]:
words_for_test = words_test_split
print(words_for_test[:20])
sm = lab01.logprob(words_for_test, alpha=0.1)
N = len(words_for_test) - 1

perplexity = math.exp(-sm/N)
print(perplexity)

['половину', 'состояния', 'давай', 'вексель', 'на', 'его', 'имя', 'ты', 'теперь', 'не', 'пьян', ',', 'в', 'своем', 'уме', ',', 'давай', ',', 'говорю', 'тебе']
3758.6911735276676


In [259]:
words_for_test = words_test_split
print(words_for_test[:20])
sm = lab01.logprob(words_for_test, alpha=0.01)
N = len(words_for_test) - 1

perplexity = math.exp(-sm/N)
print(perplexity)

['половину', 'состояния', 'давай', 'вексель', 'на', 'его', 'имя', 'ты', 'теперь', 'не', 'пьян', ',', 'в', 'своем', 'уме', ',', 'давай', ',', 'говорю', 'тебе']
2895.6819829769133


Видим, что при уменьшении параметра сглаживания перплексия становится лучше